# Week 3 (starter): Prompts as Engineering Artifacts

Runs without an API key: the semantic metric is local, and the model calls fall back to clearly-labeled fixtures so you can see the harness work. Set `GEMINI_API_KEY` to run the prompts for real. Cells marked **TODO (you)** are yours.

Dependencies: `sentence-transformers` (local). For live calls: `pip install openai` and a Gemini key.

In [1]:
import os, json, pathlib
# gemini-2.5-flash-lite is no longer available to new users swapped to 3.1-flash-lite
def gemini_chat(messages, model='gemini-3.1-flash-lite', **kw):
    """Gemini via the OpenAI-compatible endpoint. Returns text, or None if no key (API-BLOCKED)."""
    key = os.environ.get('GEMINI_API_KEY')
    if not key:
        return None
    from openai import OpenAI
    client = OpenAI(api_key=key, base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    return client.chat.completions.create(model=model, messages=messages, **kw).choices[0].message.content

LIVE = os.environ.get('GEMINI_API_KEY') is not None
print('live model calls:', LIVE, '(fixtures used when False)')

live model calls: True (fixtures used when False)


In [2]:
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer, util
emb = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
def exact_match(a, b):
    return float(str(a).strip().lower() == str(b).strip().lower())
def semantic_sim(a, b):
    e = emb.encode([a, b], convert_to_tensor=True, normalize_embeddings=True)
    return round(float(util.cos_sim(e[0], e[1])), 3)
print('metrics ready (exact-match + semantic)')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

metrics ready (exact-match + semantic)


## Part 1: Versioned prompts and a test suite
**TODO (you):** in your repo, store each prompt version as its own file. Here they are inline so the notebook runs. Task: classify a support ticket. v2 adds an intent rule.

In [3]:
PROMPT_V1 = pathlib.Path('prompt_v1.txt').read_text()
PROMPT_V2 = PROMPT_V1 + ' \n' + pathlib.Path('prompt_v2.txt').read_text()

tests = [
  {'id':1,'ticket':'I was charged twice this month, refund the duplicate.','cat':'billing','why':'duplicate charge'},
  {'id':2,'ticket':'The app crashes when I tap export.','cat':'technical','why':'crash on a feature'},
  {'id':3,'ticket':'I want to change my email but the save button does nothing.','cat':'account','why':'update profile detail'},
  {'id':4,'ticket':'Tracking has not updated in four days.','cat':'shipping','why':'delivery tracking'},
  {'id':5,'ticket':'Love the new dashboard, great work.','cat':'account','why':'feedback, no request'},
  {'id':6,'ticket':'Password reset email never arrives.','cat':'account','why':'password reset'},
  {'id':7,'ticket':'I paid for express but the box came late and crushed.','cat':'shipping','why':'delivery problem, money is context'},
  {'id':8,'ticket':'Explain the tax line on my invoice.','cat':'billing','why':'invoice question'},
  {'id':9,'ticket':'CSV import drops non-English rows.','cat':'technical','why':'import bug'},
  {'id':10,'ticket':'Close my account and delete my data.','cat':'account','why':'account closure'},
]
print('prompt versions:', 2, '| test cases:', len(tests))

prompt versions: 2 | test cases: 10


In [5]:
# Labeled fixtures stand in for model output when LIVE is False. v2 fixes #7 but regresses #3.
FIX = {
  'v1': {1:('billing','duplicate charge'),2:('technical','crash on export'),3:('account','change email'),4:('shipping','tracking'),5:('account','praise'),6:('account','reset email'),7:('billing','mentions paying'),8:('billing','invoice charge'),9:('technical','import drops rows'),10:('account','close account')},
  'v2': {1:('billing','duplicate charge'),2:('technical','crash on export'),3:('technical','save button broken'),4:('shipping','tracking'),5:('account','praise'),6:('account','reset email'),7:('shipping','late damaged delivery'),8:('billing','invoice charge'),9:('technical','import drops rows'),10:('account','close account')},
}
def run_case(version, prompt, t):
    if LIVE:
        txt = gemini_chat([{'role':'user','content': prompt + '\nTicket: ' + t['ticket']}])
        try:
            s = txt.strip().removeprefix('```json').removeprefix('```').removesuffix('```').strip()
            d = json.loads(s) 
            return d.get('category',''), d.get('rationale','')
        except Exception:
            print('failed to parse JSON:', txt, end='\n')
            return '', txt or ''
    return FIX[version][t['id']]

def score(version, prompt):
    rows = []
    for t in tests:
        cat, why = run_case(version, prompt, t)
        rows.append({'id':t['id'],'exact':exact_match(t['cat'],cat),'sem':semantic_sim(t['why'],why),'got':cat})
    acc = sum(r['exact'] for r in rows)/len(rows)
    return acc, rows

acc1, r1 = score('v1', PROMPT_V1)
acc2, r2 = score('v2', PROMPT_V2)
print(f'v1 exact-match {acc1:.0%}   v2 exact-match {acc2:.0%}')

v1 exact-match 80%   v2 exact-match 80%


## Part 3 and 4: the tradeoff and the failure
Show one case the edit improved and one it regressed. The regression is your required failure.

In [6]:
for t in tests:
    e1 = next(r for r in r1 if r['id']==t['id'])['exact']
    e2 = next(r for r in r2 if r['id']==t['id'])['exact']
    if e1 != e2:
        verdict = 'IMPROVED' if e2 > e1 else 'REGRESSED'
        print(f"#{t['id']} expected {t['cat']!r}: v1 {'ok' if e1 else 'miss'} -> v2 {'ok' if e2 else 'miss'}  [{verdict}]")
# TODO (you): explain why v2 helped one case and hurt another, and how you would resolve the tradeoff.

In [7]:
# My run did not produce the expected results, presumedly because I could not use the expected model and had to pick an alternate
# Both prompts produced the same results, yielding two errors.
def print_errors(results, tests):
    print('\nErrors:')
    for e in results:
        if e['exact'] < 1.0:
            print(tests[e['id'] -1])
            print(e, end='\n\n')

print_errors(r2, tests)


Errors:
{'id': 3, 'ticket': 'I want to change my email but the save button does nothing.', 'cat': 'account', 'why': 'update profile detail'}
{'id': 3, 'exact': 0.0, 'sem': 0.322, 'got': 'technical'}

{'id': 5, 'ticket': 'Love the new dashboard, great work.', 'cat': 'account', 'why': 'feedback, no request'}
{'id': 5, 'exact': 0.0, 'sem': 0.38, 'got': 'technical'}



\#3 is a particularly difficult example for the v2 prompt since the intention of the user is to accomplish an account change but they are encountering a technical problem

\#5 is a more straightforward example of a case not well accounted for by the prompt context

In [9]:
# A new prompt with a persona line added at the beginning to attempt improvement
PROMPT_V3 = pathlib.Path('prompt_v3.txt').read_text() + ' \n' + PROMPT_V2

print(PROMPT_V3)

acc3, r3 = score('v3', PROMPT_V3)

print(f'\nv3 exact-match {acc3:.0%}')

You are a ticket routing agent. Your goal is to route tickets to the correct department. 
Classify the ticket into one of: billing, technical, account, shipping. Return JSON {category, rationale}. 
Classify by the primary intent, not incidental words: if money is only context for a delivery problem, choose shipping.

v3 exact-match 90%


In [10]:
print_errors(r3, tests)


Errors:
{'id': 3, 'ticket': 'I want to change my email but the save button does nothing.', 'cat': 'account', 'why': 'update profile detail'}
{'id': 3, 'exact': 0.0, 'sem': 0.116, 'got': 'technical'}



Case 3 persists as an understandable failure in my opinion. The ticket does describe a technical issue and could understandably be categorized as technical by a human operator. A non-intuitive process rule as implied by this test case should be proscribed more directly as a guideline in the prompt or as a specific example case.

## Part 5: Submit
Store the prompt versions as files, run the suite (set your key for real calls), and open a pull request with the metric numbers and a linked research note. Rubric: versioned prompts (15), structured prompt (20), test suite with two metrics (25), tradeoff with numbers (25), PR hygiene (15).